# AlphaLens Event-Driven Strategy Backtest

This notebook converts the locked XGBoost test predictions into executable daily positions. Signals start at the feature-session close, expire at the 30-session target date, and never use realized labels for portfolio construction. Results include gross performance, 10 bps one-way transaction costs, turnover, drawdown, and SPY buy-and-hold.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.ml.backtest import run_event_backtest
from pipelines.ml.dataset import get_database_engine, load_adjusted_prices
from pipelines.ml.features import build_event_feature_dataset
from pipelines.ml.xgboost_model import run_xgboost_experiment

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)

## Build the locked test portfolio

Portfolio rules are fixed at top three, bottom three, at least four active company signals, and 10 bps per unit of traded notional.

In [ ]:
engine = get_database_engine()
dataset = build_event_feature_dataset(engine, horizon=30)
experiment = run_xgboost_experiment(dataset)
model_name = next(name for name in experiment.test_predictions['model'].unique() if name.startswith('xgboost_selected_'))
predictions = experiment.test_predictions.loc[experiment.test_predictions['model'] == model_name]
backtest = run_event_backtest(
    predictions,
    load_adjusted_prices(engine),
    top_k=3,
    min_signals=4,
    transaction_cost_bps=10,
)
display(backtest.summary)

## Gross, net, and benchmark wealth

In [ ]:
daily = backtest.daily_returns.set_index('trading_date').copy()
wealth = pd.DataFrame({
    'Long-short gross': (1 + daily['long_short_gross_return']).cumprod(),
    'Long-short net': (1 + daily['long_short_net_return']).cumprod(),
    'Long-only gross': (1 + daily['long_only_gross_return']).cumprod(),
    'Long-only net': (1 + daily['long_only_net_return']).cumprod(),
    'SPY': (1 + daily['spy_return']).cumprod(),
})
fig, ax = plt.subplots(figsize=(12, 5))
wealth.plot(ax=ax, linewidth=1.8)
ax.axhline(1, color='#111827', linewidth=1)
ax.set(title='Growth of one dollar on the untouched test period', xlabel='', ylabel='Portfolio value')
plt.tight_layout()
plt.show()

In [ ]:
drawdowns = wealth / wealth.cummax() - 1
fig, ax = plt.subplots(figsize=(12, 4.5))
drawdowns[['Long-short net', 'Long-only net', 'SPY']].plot(ax=ax)
ax.set(title='Drawdown from running peak', xlabel='', ylabel='Drawdown')
plt.tight_layout()
plt.show()

## Signal breadth and trading friction

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
daily['active_signals'].plot(ax=axes[0], color='#2563eb')
axes[0].axhline(4, color='#111827', linestyle='--', linewidth=1)
axes[0].set(title='Active event signals', ylabel='Companies')
daily[['long_short_turnover', 'long_only_turnover']].rolling(5).mean().plot(ax=axes[1])
axes[1].set(title='Five-session average turnover', xlabel='', ylabel='Traded notional')
plt.tight_layout()
plt.show()

In [ ]:
position_audit = pd.Series({
    'test_days': len(daily),
    'invested_days': int((daily['long_positions'] > 0).sum()),
    'unique_events_traded': backtest.weights['event_key'].nunique(),
    'unique_tickers_traded': backtest.weights['ticker'].nunique(),
    'average_active_signals': daily['active_signals'].mean(),
})
display(position_audit)
display(backtest.weights.tail(20))

## Interpretation

A backtest is an evaluation, not a promise of alpha. Gross-versus-net performance separates model quality from implementation friction. Poor untouched-test results remain part of the experiment record and must not be used to repeatedly redesign the model or portfolio until the test set looks favorable.